# Experiment 13 — Backend robustness

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** Re-generate the fixed-policy dataset with a different LLM backend
(e.g.\ GPT-4o instead of DeepSeek-reasoner) and check whether the Stage-2
selected linear policies transfer across backends.

**Data requirement.** Expects a sibling directory

```
../data/gpt4o_redo_previous_exp/results/
```

with the same file names as the default dataset. If it isn't present, this
notebook prints a skip message and exits gracefully.

In [ ]:
GPT_DIR = Path('../data/gpt4o_redo_previous_exp/results')

if not GPT_DIR.exists():
    print(f'Expected GPT-4o dataset at {GPT_DIR}; not found.')
    print('Skipping Exp 13 evaluation.')
else:
    raw2 = nu.load_all_fixed_policies(GPT_DIR)
    _, turns_gpt = nu.preprocess_turns(raw2)
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge, LinearRegression
    from sklearn.metrics import mean_absolute_error

    rows = []
    for p in POLICIES:
        feat, mdl, alpha = SELECTED[p]
        for (train_turns, train_name) in [(turns, 'deepseek'), (turns_gpt, 'gpt4o')]:
            for (test_turns, test_name) in [(turns, 'deepseek'), (turns_gpt, 'gpt4o')]:
                meta_tr, X_tr, y_tr, _ = nu.build_design_matrix(train_turns, p, feat)
                meta_te, X_te, y_te, _ = nu.build_design_matrix(test_turns,  p, feat)
                if len(X_tr) < 30 or len(X_te) < 10: continue
                scaler = StandardScaler().fit(X_tr)
                y_tr_logit = nu.logit_clip(y_tr)
                if mdl == 'OLS':
                    reg = LinearRegression().fit(scaler.transform(X_tr), y_tr_logit)
                else:
                    reg = Ridge(alpha=float(alpha) if np.isfinite(alpha) else 1.0
                                 ).fit(scaler.transform(X_tr), y_tr_logit)
                yhat = nu.sigmoid(reg.predict(scaler.transform(X_te)))
                rows.append({'policy': p, 'train': train_name, 'test': test_name,
                             'mae': float(mean_absolute_error(y_te, yhat))})
    print(pd.DataFrame(rows))